In [ ]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import warnings
import matplotlib
warnings.filterwarnings('ignore')
print("Libraries imported Successfully")
print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")
print(f"json version: {json.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")

In [ ]:
raw_df=pd.read_csv("messy_sales_data.csv")
print(f"Rows : {raw_df.shape[0]}")
print(f"Columns : {raw_df.shape[1]}")
print(f"Columns names :{raw_df.columns.tolist()}")
print("\nFirst 5 rows of raw data")
raw_df.head()

In [ ]:
print("="*55)
print("Data qualify diagnosis report")
print("="*55)
print("\n[1] MISSING VALUES per column:")
print(raw_df.isnull().sum())
print(f"\n[2] DUPLICATED VALUES:{raw_df.duplicated().sum()}")
print("\n[3] DATA TYPES")
print(raw_df.dtypes)
print("\n[4] UNIQUE CATEGORIES:",raw_df['category'].dropna().unique().tolist())
print("[4] SAMPLE ORDER DATES:",raw_df['order_date'].unique()[:8].tolist())
print("[4] SAMPLE NAMES:",raw_df['customer_name'].dropna().unique()[:8].tolist())


In [ ]:
df=raw_df.copy()
print(f"working copy created with shape:{df.shape}")
print("raw_df is untouched ")

In [ ]:
print("Before missing values",raw_df.isnull().sum().sum())
df['customer_name'].fillna('Unknown customer',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f"Filled missing quantity values with median:{median_qty}")
df['category'].fillna("Uncategorized",inplace=True)
print(f"Total missing values:{df.isnull().sum().sum()}")
df['product'].fillna('Unknown products',inplace=True)
print(f"Total missing values:{df.isnull().sum().sum()}")

In [ ]:
print(f"Rows before deduplication :{len(df)}")
print(f"Duplicate rows found :{df.duplicated().sum()}")
duplicate_mask=df.duplicated(keep=False)
print("\nThe duplicate rows (all copies shown):")
print(df[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=False))
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)
print(f"\nRows after deduplication :{len(df)}")
print(f"Rows removed :{len(raw_df)-len(df)}")




In [ ]:

print("Sample data Before parsing:")
print(df['order_date'].head(10).tolist())
df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')
nat_mask=df['order_date'].isnull()
df.loc[nat_mask,'order_date']=pd.to_datetime(raw_df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')
nat_count=df['order_date'].isnull().sum()
print(f"\n Unparseable dates remaining: {nat_count}")
df['year'] =df['order_date'].dt.year
df['month_name'] =df['order_date'].dt.strftime('%B')
df['day_name'] =df['order_date'].dt.strftime('%A')
print("\n Sample date After parsing:")
print(df[['order_date','year','month_name','day_name']].head(10).to_string())

In [ ]:
print("Names Before Standardization(sample):")
print(df['customer_name'].unique()[:20].tolist())
df['customer_name']=df['customer_name'].str.strip().str.title()
print("\nNames After Standardization(sample):")
print(df['customer_name'].unique()[:20].tolist())

In [ ]:
wrong_mask=(df['product']=='Keyboard') & (df['category']=='Electronics')
print(f"rows to fix:{wrong_mask.sum()}")
print("before fix:")
print(df[wrong_mask][['order_id','product','category']])
df.loc[wrong_mask,'category']='Computers'
print("\nafter fix:")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
df.loc[wrong_mask,'category']='Accessories'
print("\nAfter fix:")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
print("\nAll unique categories now:",sorted(df['category'].unique().tolist()))

In [ ]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print("revenue column created successfully!")
print("\nsample:quantity x unit_price=revenue")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))

In [ ]:
# ============================================================
# CELL 11 — Post-Cleaning Validation Report
# ============================================================


# Calculate the data quality score
# We check 5 things: no missing values, no duplicates, no date nulls, no revenue nulls
# Each passing check contributes 20 points (5 checks × 20 = 100)
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(df)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")

In [ ]:
output_filename='clean_sales_data.csv'
df.to_csv(output_filename,index=False)
print(f"Cleaned data saved to:{output_filename}")
print(f"Final dataset:{df.shape[0]} rows x {df.shape[1]}columns")
print("\nETL Pipeline for Sales data:COMPLETE")


In [ ]:
serp_api_key='e3fbe4dc1a0be0fa382f057c9e8806aa4058c66db4e4415df93a00f974db40a7'
serp_url='https://serpapi.com/search.json'
search_query='data engineer India'
print(f"SerpAPI Key  : {'Set (live data)' if serp_api_key != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"search query : {search_query}")

In [ ]:
# ============================================================
# CELL 22 — EXTRACT: Fetch Job Listings from SerpAPI
# ============================================================


def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(serp_url, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if serp_api_key == 'e3fbe4dc1a0be0fa382f057c9e8806aa4058c66db4e4415df93a00f974db40a7':
    print(f"Fetching job listings for: '{search_query}'")
    job_records = fetch_jobs(search_query, serp_api_key)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")

In [ ]:

import pandas as pd
df_jobs = pd.DataFrame(job_records)
csv_filename = "data_engineer_jobs_india.csv"

df_jobs.to_csv(csv_filename, index=False, encoding='utf-8')

print(f"CSV file saved successfully: {csv_filename}")
print(df_jobs.head())
print(len(df_jobs))

In [ ]:


import pandas as pd
df_jobs = pd.DataFrame(job_records)
csv_filename = "data_engineer_jobs_india.csv"

df_jobs.to_csv(
    csv_filename,
    index=False,
    encoding='utf-8'
)

print(f"CSV file saved successfully: {csv_filename}")

from google.colab import files

files.download(csv_filename)